# Modèle final de gravité automobile — XGBoost Gamma unique

**Modèle retenu : quatre variables, découpage 80/20 initial par client.** La version sauvegardée est ajustée uniquement sur les **2 356 sinistres positifs d'apprentissage** ; les **561 sinistres de validation** restent séparés. Aucun modèle de fréquence n'entre dans cette version.

**RMSE validation 20 % : 1 463,81 € ; MAE : 930,49 €.** La RMSE hors pli des trois plis par client du développement est de **1 448,34 €**, selon un autre protocole. Les données ayant déjà été explorées, ces évaluations ne constituent pas une validation externe neuve.

Les fichiers de référence sont `models/gravite_final.joblib` et `models/gravite_final.json`. Le script `predict_final.py` produit les coûts conditionnels. Cette synthèse publique contient du code, des comptes et des scores agrégés ; les découpages et prédictions par contrat sont générés localement, exclus du suivi Git et non inclus dans les sorties affichées.

Le notebook de recherche complet est conservé dans le travail local ; cette synthèse permet de reproduire le modèle retenu sans publier les rapports individuels.

## 1. Cadrage et roadmap

![Roadmap](assets/roadmap_reference.png)

La cible est le coût conditionnel **E[montant_sinistre | montant_sinistre > 0, X]**, sans multiplication par une probabilité de fréquence.

La démarche suit le cadrage, la description de la cible et des variables, le traitement des anomalies, le choix et la création des variables, la comparaison de modèles, la validation par client et la sauvegarde. La recherche a comparé plusieurs représentations de véhicules, de conducteurs, de contrats et de géographie, ainsi que des modèles par tranches. Le modèle unique est retenu pour sa simplicité et l'absence de gain stable des segmentations. Les recettes et les scores antérieurs ne sont pas présentés comme des mesures externes indépendantes.

In [1]:
from pathlib import Path
import os, sys, json, time, hashlib, platform, inspect, warnings
os.environ.setdefault('MPLCONFIGDIR', '/tmp/gravite_mpl')
os.environ.setdefault('OMP_NUM_THREADS', '2')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '2')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Code, Image
import scipy, sklearn, xgboost, joblib
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, GroupKFold, ParameterSampler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_gamma_deviance, r2_score
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import GammaRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from threadpoolctl import threadpool_limits

WORKING_DIR = Path.cwd().resolve()
CANDIDATES = [WORKING_DIR, WORKING_DIR / 'gravite_xgboost',
              WORKING_DIR / 'TD-Tarification-automobile-par-Machine-Learning' / 'gravite_xgboost']
HERE = next((p for p in CANDIDATES if (p / 'gravite_pipeline.py').exists()), None)
assert HERE is not None, 'Ouvrir depuis le dossier Project, la racine du dépôt ou ce dossier.'
sys.path.insert(0, str(HERE))
from gravite_pipeline import SeverityFeatures, SeverityEstimator, FORBIDDEN, GROUPS
DATA = HERE.parent / 'Data'
REPORTS, MODELS = HERE / 'reports', HERE / 'models'
REPORTS.mkdir(exist_ok=True); MODELS.mkdir(exist_ok=True)
SEED = 2026
np.random.seed(SEED)
pd.set_option('display.max_rows', 40)
pd.set_option('display.max_columns', 16)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': .18, 'axes.prop_cycle': matplotlib.cycler(color=['#215e80','#c67538','#5e8e75'])})
VERSIONS = {m.__name__: m.__version__ for m in [np,pd,scipy,sklearn,xgboost,matplotlib,joblib]}
VERSIONS['python'] = platform.python_version()
print('Versions réellement utilisées :', VERSIONS)
START = time.time()

def savefig(name):
    plt.tight_layout()
    plt.savefig(REPORTS / (name + '.png'), bbox_inches='tight')
    display(Image(filename=str(REPORTS / (name + '.png'))))
    plt.close()

def note(text): display(Markdown(text))
def rmse(y, p): return float(np.sqrt(mean_squared_error(y, p)))
def scores(y, p):
    y, p = np.asarray(y), np.maximum(np.asarray(p), 1.)
    return {'rmse': rmse(y,p), 'mae': float(mean_absolute_error(y,p)),
            'gamma_deviance': float(mean_gamma_deviance(y,p)), 'r2': float(r2_score(y,p)) if len(y)>1 else np.nan,
            'ratio_predit_observe': float(p.sum()/y.sum()), 'moyenne_predite': float(p.mean())}

/home/cytech/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Versions réellement utilisées : {'numpy': '2.5.3', 'pandas': '3.0.6', 'scipy': '1.18.1', 'sklearn': '1.9.1', 'xgboost': '3.0.2', 'matplotlib': '3.11.2', 'joblib': '1.6.0', 'python': '3.12.3'}


## 2. Données et découpage 80/20 par client

On conserve le découpage initial : `GroupShuffleSplit(test_size=0.20, random_state=2026)` sur le portefeuille complet. Il sépare 39 964 contrats d'apprentissage et 10 036 contrats de validation. Les 80/20 concernent les clients ; parmi les seuls sinistrés, cela donne 2 356 et 561 coûts positifs.

Les identifiants servent uniquement au découpage et au rapprochement des sorties. Ils ne sont pas des variables prédictives. Les cibles et le sexe des conducteurs sont également exclus. Les montants nuls sont écartés de l'apprentissage de gravité. Un montant négatif ou une incohérence entre sinistre et coût arrêterait l'exécution : il n'est pas corrigé silencieusement.

In [2]:
train = pd.read_csv(DATA / 'train.csv', dtype={'code_postal': str})
test = pd.read_csv(DATA / 'test.csv', dtype={'code_postal': str})
TARGET = 'montant_sinistre'
assert train['index'].is_unique and test['index'].is_unique
assert train[TARGET].notna().all() and train[TARGET].ge(0).all()
assert train['nombre_sinistres'].notna().all() and train['nombre_sinistres'].ge(0).all()
assert (train['nombre_sinistres'].gt(0) == train[TARGET].gt(0)).all(), 'Cible/fréquence incohérentes : à résoudre.'
FEATURES = [c for c in test.columns if c not in FORBIDDEN]
assert set(FEATURES).issubset(train.columns)
assert not set(FEATURES) & FORBIDDEN

splitter = GroupShuffleSplit(n_splits=1, test_size=.20, random_state=SEED)
dev_idx, hold_idx = next(splitter.split(train, groups=train['id_client']))
dev_portfolio = train.iloc[dev_idx].copy()
hold_portfolio = train.iloc[hold_idx].copy()
assert not set(dev_portfolio.id_client) & set(hold_portfolio.id_client)
dev = dev_portfolio.loc[dev_portfolio.nombre_sinistres.gt(0)].copy().reset_index(drop=True)
# Les cibles de validation ne servent pas à ajuster le modèle final.
X, y, groups = dev[FEATURES], dev[TARGET].to_numpy(), dev.id_client.to_numpy()
strata = pd.qcut(pd.Series(y).rank(method='first'), q=5, labels=False)
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
FOLDS = list(cv.split(X, strata, groups))
fold_ids = np.full(len(dev), -1)
for k,(tr,va) in enumerate(FOLDS):
    assert not set(groups[tr]) & set(groups[va])
    fold_ids[va] = k
assert (fold_ids >= 0).all()
assign = train[['index','id_client']].copy()
assign['role'] = np.where(assign.index.isin(dev_idx), 'developpement', 'holdout')
assign['pli_cv_gravite'] = assign['index'].map(dict(zip(dev['index'],fold_ids))).astype('Int64')
assign.to_csv(REPORTS / 'decoupage_clients.csv', index=False)
summary_files = pd.DataFrame([
    {'fichier':'train.csv','lignes':len(train),'colonnes':train.shape[1],'rôle':'apprentissage + holdout local'},
    {'fichier':'test.csv','lignes':len(test),'colonnes':test.shape[1],'rôle':'prédictions finales sans cibles'},
])
display(summary_files)
print(f'Développement : {len(dev_portfolio):,} contrats ; {len(dev):,} sinistrés ; {len(np.unique(groups)):,} clients sinistrés.')
print(f'Holdout réservé : {len(hold_portfolio):,} contrats, clients séparés de l’apprentissage.')
print('Taille des plis positifs :', pd.Series(fold_ids).value_counts().sort_index().to_dict())

,fichier,lignes,colonnes,rôle
0,train.csv,50000,33,apprentissage + holdout local
1,test.csv,50000,28,prédictions finales sans cibles


Développement : 39,964 contrats ; 2,356 sinistrés ; 2,338 clients sinistrés.
Holdout réservé : 10,036 contrats, clients séparés de l’apprentissage.
Taille des plis positifs : {0: 785, 1: 786, 2: 785}


In [3]:
positive_amounts=train.loc[train[TARGET].gt(0),TARGET]
display(pd.DataFrame([{
    'contrats':len(train),'montants_nuls':int(train[TARGET].eq(0).sum()),
    'sinistres_positifs':len(positive_amounts),'minimum_positif':float(positive_amounts.min()),
    'mediane_positive':float(positive_amounts.median()),'moyenne_positive':float(positive_amounts.mean()),
    'quantile_95':float(positive_amounts.quantile(.95)),'maximum':float(positive_amounts.max())}]))

,contrats,montants_nuls,sinistres_positifs,minimum_positif,mediane_positive,moyenne_positive,quantile_95,maximum
0,50000,47083,2917,600.600,"1,236.000","1,771.843","4,355.258","21,826.960"


## 3. Choix et construction des quatre variables

- **`type_contrat`** : catégorie observée (`Maxi`, `Median1`, `Median2`, `Mini`), encodée en indicatrices 0/1. Elle renseigne la couverture associée au montant enregistré.
- **`prix_vehicule`** : valeur numérique du dataset, conservée sans logarithme ni standardisation. Une valeur manquante est remplacée par la médiane du seul sous-train.
- **`anciennete_vehicule`** : ancienneté fournie dans le dataset, sans recalcul depuis les dates de commercialisation. Les manquants sont complétés par la médiane du seul sous-train.
- **`classe_geographique`** : rapprochement de `code_postal`, conservé en texte sur cinq caractères, avec la grille INSEE : urbain dense, urbain intermédiaire, rural ou inconnu. Les arrondissements de Paris, Lyon et Marseille sont rattachés à leur commune. La taille de commune a été retirée.

Les codes sont interprétés comme des codes communaux INSEE au vu des valeurs observées, malgré le nom `code_postal` ; cette interprétation reste une hypothèse à confirmer auprès du producteur. La géographie du référentiel est celle de 2021 et sa population de référence celle de 2018. Les dates des contrats étant inconnues, la compatibilité temporelle n'est pas garantie. [Source INSEE](https://www.insee.fr/fr/information/6439600).

Les regroupements et les variables véhicule supplémentaires ont été comparés dans la recherche ; une justification métier ne suffit pas à démontrer leur utilité prédictive. La version finale conserve les quatre variables sélectionnées. Les transformations réutilisables sont dans `qualitative_features.py`, `metier_features.py` et `gravite_pipeline.py`.

## 4. Modèle retenu et validation croisée du développement

Le modèle est un **XGBoost Gamma à lien logarithmique** : 400 arbres, profondeur 1, taux 0,03, `min_child_weight=40`, `reg_lambda=20`, `reg_alpha=0`, `subsample=0.9`, `colsample_bytree=1`. Ces réglages sont figés ; aucune nouvelle optimisation n'est faite sur la validation.

La validation croisée réutilise les trois plis par client du développement. Chaque modèle, ses médianes et son encodage sont ajustés uniquement sur les sous-trains. Ce score a servi aux comparaisons de la recherche et reste descriptif ; il n'est pas le score du jeu de validation 20 %.

**Limite de l’optimisation :** la recherche initiale a comparé plusieurs configurations, puis les expériences Gamma ont principalement opposé deux niveaux de complexité. Il ne s’agit pas d’une optimisation exhaustive dédiée à la version finale à quatre variables. Les paramètres restent figés pour cette évaluation 80/20.

In [4]:
from qualitative_features import QualitativeSeverityEstimator
recipe=json.loads((MODELS/'gravite_final.json').read_text())['configuration']
public_oof=np.empty(len(y)); public_fold_scores=[]
for fold,(tr,va) in enumerate(FOLDS):
    assert not set(groups[tr])&set(groups[va])
    with threadpool_limits(limits=2):
        fitted=QualitativeSeverityEstimator(recipe['spec'],recipe['params'],recipe['objective']).fit(X.iloc[tr],y[tr])
    public_oof[va]=fitted.predict(X.iloc[va])
    public_fold_scores.append({'pli':fold,'n_sinistres':len(va),**scores(y[va],public_oof[va])})
reference_cv=pd.read_csv(REPORTS/'final/cv_developpement.csv').iloc[0]
np.testing.assert_allclose(rmse(y,public_oof),reference_cv.rmse,rtol=0,atol=1e-6)
display(pd.DataFrame(public_fold_scores)[['pli','n_sinistres','rmse','mae']])
display(pd.DataFrame([{'protocole':'hors pli developpement','n_sinistres':len(y),**scores(y,public_oof)}]))

,pli,n_sinistres,rmse,mae
0,0,785,"1,317.627",898.709
1,1,786,"1,590.679",941.728
2,2,785,"1,423.416",932.547


,protocole,n_sinistres,rmse,mae,gamma_deviance,r2,ratio_predit_observe,moyenne_predite
0,hors pli developpement,2356,"1,448.340",924.335,0.396,0.018,0.995,"1,758.936"


## 5. Ajustement final sur les 80 %, évaluation sur les 20 % et sauvegarde

Le modèle final est ajusté sur les seuls 2 356 coûts positifs d'apprentissage. La validation comprend 561 coûts positifs de clients distincts. Aucun réajustement sur ces 561 lignes n'est réalisé après la mesure. Les médianes et l'encodage sont donc appris uniquement sur les 80 %.

Le modèle canonique, son manifeste et les comptes agrégés sont versionnés. Les CSV de prédictions et de découpage sont produits pour l'usage local et exclus de la publication. La sauvegarde est contrôlée par rechargement, reproduction des prédictions et invariance aux cibles fournies lors de la prédiction et à la population communale retirée.

In [5]:
from qualitative_features import QualitativeSeverityEstimator
import copy
FINAL_REPORTS=REPORTS/'final'; FINAL_REPORTS.mkdir(exist_ok=True)
f_parent=json.loads((MODELS/'gravite_final.json').read_text())
f_config=copy.deepcopy(f_parent['configuration'])
f_train=dev.copy()
f_validation=hold_portfolio.loc[hold_portfolio[TARGET].gt(0)].copy()
assert len(f_train)==2356 and len(f_validation)==561
assert not set(f_train.id_client)&set(f_validation.id_client)
assert not set(f_train['index'])&set(f_validation['index'])
assert f_train[TARGET].gt(0).all() and f_validation[TARGET].gt(0).all()
with threadpool_limits(limits=2):
    f_model=QualitativeSeverityEstimator(f_config['spec'],f_config['params'],f_config['objective']).fit(
        f_train[FEATURES],f_train[TARGET])
assert f_model.feature_columns_==['type_contrat','prix_vehicule','anciennete_vehicule','classe_geographique']
f_train_pred=f_model.predict(f_train[FEATURES])
f_validation_pred=f_model.predict(f_validation[FEATURES])
f_test_pred=f_model.predict(test[FEATURES])
assert len(f_test_pred)==len(test) and np.isfinite(f_test_pred).all() and (f_test_pred>0).all()
f_training_scores=scores(f_train[TARGET],f_train_pred)
f_validation_scores=scores(f_validation[TARGET],f_validation_pred)
f_cv_table=pd.read_csv(REPORTS/'final/cv_developpement.csv')
f_cv_row=f_cv_table.loc[f_cv_table.version.eq('sans_taille_commune')].iloc[0]
f_cv_scores={k:float(f_cv_row[k]) for k in ['rmse','mae','gamma_deviance','r2','ratio_predit_observe','moyenne_predite']}
f_score_table=pd.DataFrame([
    {'protocole':'apprentissage_80','n_sinistres':len(f_train),**f_training_scores},
    {'protocole':'validation_20','n_sinistres':len(f_validation),**f_validation_scores},
    {'protocole':'validation_croisee_3_plis_developpement','n_sinistres':len(f_train),**f_cv_scores}])
f_score_table.to_csv(FINAL_REPORTS/'scores.csv',index=False)
f_counts=pd.DataFrame([{'jeu':name,'contrats_portefeuille':len(portfolio),'clients_portefeuille':portfolio.id_client.nunique(),
    'sinistres_positifs':len(positive),'clients_sinistres':positive.id_client.nunique()}
    for name,portfolio,positive in [('apprentissage_80',dev_portfolio,f_train),('validation_20',hold_portfolio,f_validation)]])
f_counts.to_csv(FINAL_REPORTS/'effectifs_80_20.csv',index=False)
(FINAL_REPORTS/'decoupage_clients_80_20.csv').write_bytes((REPORTS/'decoupage_clients.csv').read_bytes())
pd.DataFrame({'index':f_validation['index'].to_numpy(),'id_client':f_validation.id_client.to_numpy(),
    'montant_observe':f_validation[TARGET].to_numpy(),'gravite_predite':f_validation_pred}).to_csv(
        FINAL_REPORTS/'predictions_validation.csv',index=False)
pd.DataFrame({'index':test['index'].to_numpy(),'gravite_predite':f_test_pred}).to_csv(FINAL_REPORTS/'gravite_test.csv',index=False)
f_path=MODELS/'gravite_final.joblib'; joblib.dump(f_model,f_path)
f_reloaded=joblib.load(f_path)
np.testing.assert_allclose(f_reloaded.predict(f_validation[FEATURES]),f_validation_pred,rtol=0,atol=1e-8)
np.testing.assert_allclose(f_reloaded.predict(test[FEATURES]),f_test_pred,rtol=0,atol=1e-8)
f_probe=f_validation[FEATURES].head(100).copy(); f_probe[TARGET]=-1.e9; f_probe['nombre_sinistres']=999
np.testing.assert_array_equal(f_reloaded.predict(f_probe),f_validation_pred[:len(f_probe)])
f_changed=copy.deepcopy(f_reloaded)
f_changed.pipeline_.named_steps['qualitative'].geo_.reference_['population_2018']=1.e9
np.testing.assert_array_equal(f_changed.predict(f_validation[FEATURES].head(100)),f_validation_pred[:100])
f_geo=json.loads((HERE/'assets/source_geographie.json').read_text())
f_manifest={'status':'modele_final_retenu','model_family':'XGBoost Gamma unique',
    'target':'E[montant_sinistre | montant_sinistre > 0, X]',
    'model_path':str(f_path.relative_to(HERE)),
    'model_sha256':hashlib.sha256(f_path.read_bytes()).hexdigest(),
    'configuration':f_config,'features':f_model.feature_columns_,
    'qualitative_features':f_model.qualitative_columns_,
    'encoded_features':f_model.pipeline_.named_steps['encoding'].get_feature_names_out().tolist(),
    'raw_input_columns':FEATURES,
    'split':{'method':'GroupShuffleSplit sur le portefeuille complet','random_state':SEED,'test_size_clients':.20,
        'training_portfolio_rows':len(dev_portfolio),'validation_portfolio_rows':len(hold_portfolio),
        'training_positive_rows':len(f_train),'validation_positive_rows':len(f_validation),
        'training_positive_clients':f_train.id_client.nunique(),'validation_positive_clients':f_validation.id_client.nunique(),
        'client_overlap':False,'source':'reports/final/decoupage_clients_80_20.csv',
        'sha256':hashlib.sha256((FINAL_REPORTS/'decoupage_clients_80_20.csv').read_bytes()).hexdigest()},
    'fitted_on_training_80_only':True,'validation_rows_used_for_fit':False,
    'metrics':{'training_80':f_training_scores,'validation_20':f_validation_scores,
        'development_cv_3_folds':f_cv_scores},
    'data_sha256':{name:hashlib.sha256((DATA/name).read_bytes()).hexdigest() for name in ['train.csv','test.csv']},
    'source_sha256':{name:hashlib.sha256((HERE/name).read_bytes()).hexdigest() for name in
        ['gravite_pipeline.py','metier_features.py','qualitative_features.py','predict_final.py']},
    'geography_reference':f_geo,'code_postal_interpreted_as_insee_commune_code':True,
    'geography_temporal_compatibility_unknown':True,
    'validation_data_previously_explored':True,'external_validation':False,
    'threshold_or_hyperparameters_retuned':False,'frequency_model_trained':False,
    'frequency_encodings_used':False,'n_test_predictions':len(f_test_pred),
    'test_predictions_path':'reports/final/gravite_test.csv','reload_verified':True,
    'prediction_target_invariance_verified':True,'population_invariance_verified':True,
    'selection_reason':'Modele unique choisi pour sa simplicite et l’absence de gain stable des modeles par tranches.',
    'versions':VERSIONS}
(MODELS/'gravite_final.json').write_text(json.dumps(f_manifest,indent=2,ensure_ascii=False))
(FINAL_REPORTS/'manifest.json').write_text(json.dumps(f_manifest,indent=2,ensure_ascii=False))
display(f_counts); display(f_score_table[['protocole','n_sinistres','rmse','mae','ratio_predit_observe']])
print(f'MODELE FINAL UNIQUE 80/20 : {len(f_train)} sinistres apprentissage ; {len(f_validation)} validation.',flush=True)
print(f'RMSE validation 20 % : {f_validation_scores["rmse"]:.6f} euros ; MAE : {f_validation_scores["mae"]:.6f} euros.',flush=True)
note(f'**Modèle final publié : XGBoost Gamma unique à quatre variables.** '
     f'Apprentissage sur **{len(f_train):,}** sinistres positifs ; validation séparée sur **{len(f_validation):,}** sinistres. '
     f'**RMSE de validation 80/20 : {f_validation_scores["rmse"]:,.2f} €** ; '
     f'MAE : **{f_validation_scores["mae"]:,.2f} €**. '
     f'La RMSE hors pli du développement reste **{f_cv_scores["rmse"]:,.2f} €** selon un autre protocole. '
     'Le modèle final n’a pas été réentraîné sur la validation. '
     'Les 50 000 prédictions ont été vérifiées après rechargement ; les essais par tranches restent historiques.')


,jeu,contrats_portefeuille,clients_portefeuille,sinistres_positifs,clients_sinistres
0,apprentissage_80,39964,36628,2356,2338
1,validation_20,10036,9157,561,558


,protocole,n_sinistres,rmse,mae,ratio_predit_observe
0,apprentissage_80,2356,"1,438.562",918.929,0.997
1,validation_20,561,"1,463.806",930.494,0.984
2,validation_croisee_3_plis_developpement,2356,"1,448.340",924.335,0.995


MODELE FINAL UNIQUE 80/20 : 2356 sinistres apprentissage ; 561 validation.


RMSE validation 20 % : 1463.806037 euros ; MAE : 930.493754 euros.


**Modèle final publié : XGBoost Gamma unique à quatre variables.** Apprentissage sur **2,356** sinistres positifs ; validation séparée sur **561** sinistres. **RMSE de validation 80/20 : 1,463.81 €** ; MAE : **930.49 €**. La RMSE hors pli du développement reste **1,448.34 €** selon un autre protocole. Le modèle final n’a pas été réentraîné sur la validation. Les 50 000 prédictions ont été vérifiées après rechargement ; les essais par tranches restent historiques.

## 6. Décision et limites

Le **modèle unique en 80/20** est la version finale choisie. Les essais par tranches n'ont pas démontré un gain stable : sur trois plis du développement, le modèle unique obtient 1 448,34 € contre 1 450,00 € pour deux tranches. Les petits écarts du découpage historique 90/10 concernent un autre jeu de validation et ne sont pas les résultats du modèle final 80/20.

Les gros sinistres restent difficiles à prévoir. La géographie est un enrichissement externe avec des limites de codification et de millésime. Les données ont déjà été consultées au cours de plusieurs expériences ; aucune validation externe neuve n'est revendiquée.

Pour prédire sans réentraîner : `python predict_final.py --input ../Data/test.csv --output /tmp/gravite_predictions.csv`. Les valeurs produites sont des gravités conditionnelles, pas des primes. Pour reproduire cette synthèse : `python run_final_notebook.py`.